In [1]:
# ============================================================
# CELL 1 - IMPORTS
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from datetime import datetime

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2 - PROJECT PATHS
# ============================================================

PROJECT_ROOT = ".."

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data"
)

PROCESSED_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

FEATURE_DIR = os.path.join(
    DATA_DIR,
    "features"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

print("Project root:")
print(os.path.abspath(PROJECT_ROOT))

print("\nModel directory:")
print(os.path.abspath(MODEL_DIR))

print("\nFeature directory:")
print(os.path.abspath(FEATURE_DIR))

Project root:
c:\Users\User\Documents\GitHub\LankaLandslide-AI

Model directory:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\models

Feature directory:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\features


In [3]:
# ============================================================
# CELL 3 - CHECK REQUIRED DEPLOYMENT FILES
# ============================================================

required_files = [
    "xgboost_landslide_model.pkl",
    "logistic_landslide_model.pkl",
    "random_forest_landslide_model.pkl",
    "feature_columns.pkl",
    "model_thresholds.pkl",
    "lankalandslide_final_model.pkl",
    "deployment_feature_columns.json",
    "deployment_config.json"
]

print("=" * 70)
print("DEPLOYMENT FILE CHECK")
print("=" * 70)

missing_files = []

for filename in required_files:

    path = os.path.join(
        MODEL_DIR,
        filename
    )

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            f"✓ {filename:<45} "
            f"{size_mb:.2f} MB"
        )

    else:
        print(
            f"✗ {filename:<45} MISSING"
        )

        missing_files.append(filename)

print("\n" + "=" * 70)

if len(missing_files) == 0:
    print("All required deployment files are available.")
else:
    print("Missing files:")
    for file in missing_files:
        print("-", file)

DEPLOYMENT FILE CHECK
✓ xgboost_landslide_model.pkl                   1.12 MB
✓ logistic_landslide_model.pkl                  0.00 MB
✓ random_forest_landslide_model.pkl             63.38 MB
✓ feature_columns.pkl                           0.00 MB
✓ model_thresholds.pkl                          0.00 MB
✓ lankalandslide_final_model.pkl                1.12 MB
✓ deployment_feature_columns.json               0.00 MB
✓ deployment_config.json                        0.00 MB

All required deployment files are available.


In [4]:
# ============================================================
# CELL 4 - LOAD FINAL XGBOOST MODEL
# ============================================================

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "lankalandslide_final_model.pkl"
)

final_model = joblib.load(
    FINAL_MODEL_PATH
)

print("Final model loaded successfully.")
print()

print("Model type:")
print(type(final_model))

print()

print("Model:")
print(final_model)

Final model loaded successfully.

Model type:
<class 'xgboost.sklearn.XGBClassifier'>

Model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)


In [5]:
# ============================================================
# CELL 5 - LOAD EXACT TRAINING FEATURES
# ============================================================

FEATURE_COLUMNS_PATH = os.path.join(
    MODEL_DIR,
    "feature_columns.pkl"
)

feature_columns = joblib.load(
    FEATURE_COLUMNS_PATH
)

print("=" * 70)
print("FEATURE CONFIGURATION")
print("=" * 70)

print("Number of features:", len(feature_columns))

print()

for index, feature in enumerate(
    feature_columns,
    start=1
):
    print(
        f"{index:02d}. {feature}"
    )

FEATURE CONFIGURATION
Number of features: 27

01. rainfall_1day
02. rainfall_3day
03. rainfall_7day
04. rainfall_14day
05. rainfall_30day
06. rainfall_3day_avg
07. rainfall_7day_avg
08. rainfall_14day_avg
09. rainfall_30day_avg
10. rain_hours
11. temp_max
12. temp_min
13. temp_mean
14. temperature_range
15. wind_max
16. wind_gust_max
17. month_sin
18. month_cos
19. heavy_rain_10mm
20. heavy_rain_25mm
21. heavy_rain_50mm
22. heavy_rain_100mm
23. elevation_mean_m
24. elevation_max_m
25. elevation_min_m
26. slope_mean_deg
27. slope_max_deg


In [6]:
# ============================================================
# CELL 6 - VERIFY FEATURE CONFIGURATION
# ============================================================

JSON_FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "deployment_feature_columns.json"
)

with open(
    JSON_FEATURE_PATH,
    "r"
) as f:

    deployment_features = json.load(f)

print("Original feature count:")
print(len(feature_columns))

print("\nDeployment feature count:")
print(len(deployment_features))

print()

if feature_columns == deployment_features:

    print("✓ FEATURE CONFIGURATION MATCHES")

else:

    print("✗ FEATURE CONFIGURATION DOES NOT MATCH")

    print("\nOriginal:")
    print(feature_columns)

    print("\nDeployment:")
    print(deployment_features)

    raise ValueError(
        "Feature configuration mismatch."
    )

Original feature count:
27

Deployment feature count:
27

✓ FEATURE CONFIGURATION MATCHES


In [7]:
# ============================================================
# CELL 7 - LOAD DEPLOYMENT CONFIGURATION
# ============================================================

CONFIG_PATH = os.path.join(
    MODEL_DIR,
    "deployment_config.json"
)

with open(
    CONFIG_PATH,
    "r"
) as f:

    deployment_config = json.load(f)

print(
    json.dumps(
        deployment_config,
        indent=4
    )
)

{
    "project": "LankaLandslide AI",
    "model_name": "XGBoost",
    "model_type": "XGBClassifier",
    "model_file": "lankalandslide_final_model.pkl",
    "feature_file": "deployment_feature_columns.json",
    "threshold": 0.49,
    "prediction_unit": "district-day",
    "target": "landslide_recorded",
    "training_period": "1998-01-01 to 2019-12-31",
    "test_period": "2018-01-01 to 2019-12-31",
    "note": "This model estimates district-level landslide risk from weather and terrain features. It is a decision-support model and is not an official warning system."
}


In [8]:
# ============================================================
# CELL 8 - GET FINAL XGBOOST THRESHOLD
# ============================================================

threshold = float(
    deployment_config.get(
        "threshold",
        0.50
    )
)

print("Deployment threshold:")
print(threshold)

Deployment threshold:
0.49


In [9]:
# ============================================================
# CELL 9 - LOAD FINAL FEATURE DATASET
# ============================================================

FEATURE_DATA_PATH = os.path.join(
    FEATURE_DIR,
    "landslide_weather_terrain_features.csv"
)

df_features = pd.read_csv(
    FEATURE_DATA_PATH
)

print("Dataset loaded successfully.")

print()
print("Shape:")
print(df_features.shape)

print()
print("Columns:")
print(df_features.columns.tolist())

Dataset loaded successfully.

Shape:
(192120, 37)

Columns:
['date', 'rainfall_mm', 'rain_mm', 'rain_hours', 'temp_max', 'temp_min', 'temp_mean', 'wind_max', 'wind_gust_max', 'district', 'latitude', 'longitude', 'landslide_events', 'landslide_recorded', 'rainfall_1day', 'rainfall_3day', 'rainfall_7day', 'rainfall_14day', 'rainfall_30day', 'rainfall_3day_avg', 'rainfall_7day_avg', 'rainfall_14day_avg', 'rainfall_30day_avg', 'temperature_range', 'month', 'day_of_year', 'month_sin', 'month_cos', 'heavy_rain_10mm', 'heavy_rain_25mm', 'heavy_rain_50mm', 'heavy_rain_100mm', 'elevation_mean_m', 'elevation_max_m', 'elevation_min_m', 'slope_mean_deg', 'slope_max_deg']


In [10]:
# ============================================================
# CELL 10 - VERIFY MODEL FEATURES EXIST IN DATASET
# ============================================================

missing_features = [
    feature
    for feature in feature_columns
    if feature not in df_features.columns
]

extra_features = [
    column
    for column in df_features.columns
    if column not in feature_columns
]

print("=" * 70)
print("FEATURE VALIDATION")
print("=" * 70)

print("\nRequired model features:")
print(len(feature_columns))

print("\nMissing model features:")
print(len(missing_features))

if missing_features:

    for feature in missing_features:
        print("MISSING:", feature)

    raise ValueError(
        "Some model features are missing from the deployment dataset."
    )

else:

    print("✓ All model features exist.")

print("\nAdditional dataset columns:")
print(len(extra_features))

for column in extra_features:
    print("-", column)

FEATURE VALIDATION

Required model features:
27

Missing model features:
0
✓ All model features exist.

Additional dataset columns:
10
- date
- rainfall_mm
- rain_mm
- district
- latitude
- longitude
- landslide_events
- landslide_recorded
- month
- day_of_year


In [11]:
# ============================================================
# CELL 11 - CREATE MODEL INPUT MATRIX
# ============================================================

X_all = df_features[
    feature_columns
].copy()

print("X shape:")
print(X_all.shape)

print()

print("Missing values:")
print(
    X_all.isnull().sum().sum()
)

print()

print("Data types:")
print(
    X_all.dtypes
)

X shape:
(192120, 27)

Missing values:
0

Data types:
rainfall_1day         float64
rainfall_3day         float64
rainfall_7day         float64
rainfall_14day        float64
rainfall_30day        float64
rainfall_3day_avg     float64
rainfall_7day_avg     float64
rainfall_14day_avg    float64
rainfall_30day_avg    float64
rain_hours            float64
temp_max              float64
temp_min              float64
temp_mean             float64
temperature_range     float64
wind_max              float64
wind_gust_max         float64
month_sin             float64
month_cos             float64
heavy_rain_10mm         int64
heavy_rain_25mm         int64
heavy_rain_50mm         int64
heavy_rain_100mm        int64
elevation_mean_m      float64
elevation_max_m       float64
elevation_min_m       float64
slope_mean_deg        float64
slope_max_deg         float64
dtype: object


In [12]:
# ============================================================
# CELL 12 - VERIFY NUMERIC MODEL INPUT
# ============================================================

non_numeric_features = []

for column in feature_columns:

    if not pd.api.types.is_numeric_dtype(
        X_all[column]
    ):

        non_numeric_features.append(
            column
        )

if non_numeric_features:

    print("Non-numeric features found:")

    for feature in non_numeric_features:
        print("-", feature)

    raise TypeError(
        "Deployment model expects numeric features."
    )

else:

    print(
        "✓ All deployment features are numeric."
    )

✓ All deployment features are numeric.


In [13]:
# ============================================================
# CELL 13 - TEST XGBOOST PROBABILITY PREDICTION
# ============================================================

test_sample = X_all.iloc[
    0:5
]

test_probabilities = final_model.predict_proba(
    test_sample
)[:, 1]

print("Test probabilities:")

for index, probability in enumerate(
    test_probabilities,
    start=1
):

    print(
        f"Sample {index}: "
        f"{probability:.6f}"
    )

Test probabilities:
Sample 1: 0.005526
Sample 2: 0.006074
Sample 3: 0.002184
Sample 4: 0.000951
Sample 5: 0.000813


In [14]:
# ============================================================
# CELL 14 - RISK CLASSIFICATION FUNCTION
# ============================================================

def classify_risk(
    probability,
    threshold
):

    if probability >= threshold:

        return "HIGH"

    else:

        return "LOW"


print("Risk classification function created.")

Risk classification function created.


In [15]:
# ============================================================
# CELL 14 - RISK CLASSIFICATION FUNCTION
# ============================================================

def classify_risk(
    probability,
    threshold
):

    if probability >= threshold:

        return "HIGH"

    else:

        return "LOW"


print("Risk classification function created.")

Risk classification function created.


In [16]:
# ============================================================
# CELL 15 - TEST RISK CLASSIFICATION
# ============================================================

for probability in test_probabilities:

    risk = classify_risk(
        probability,
        threshold
    )

    print(
        f"Probability: {probability:.4f} "
        f"→ Risk: {risk}"
    )

Probability: 0.0055 → Risk: LOW
Probability: 0.0061 → Risk: LOW
Probability: 0.0022 → Risk: LOW
Probability: 0.0010 → Risk: LOW
Probability: 0.0008 → Risk: LOW


In [17]:
# ============================================================
# CELL 16 - COMPLETE DEPLOYMENT PREDICTION FUNCTION
# ============================================================

def predict_landslide_risk(
    input_data,
    model=final_model,
    features=feature_columns,
    threshold=threshold
):
    """
    Generate a landslide-risk prediction.

    Parameters
    ----------
    input_data : dict
        Dictionary containing the exact model features.

    model : trained model
        Final XGBoost model.

    features : list
        Exact feature order used during training.

    threshold : float
        Classification threshold.

    Returns
    -------
    dict
        Probability and risk classification.
    """

    # --------------------------------------------------------
    # Validate required features
    # --------------------------------------------------------

    missing = [
        feature
        for feature in features
        if feature not in input_data
    ]

    if missing:

        raise ValueError(
            "Missing required features: "
            + ", ".join(missing)
        )

    # --------------------------------------------------------
    # Create DataFrame
    # --------------------------------------------------------

    input_df = pd.DataFrame(
        [
            input_data
        ]
    )

    # --------------------------------------------------------
    # Ensure exact feature order
    # --------------------------------------------------------

    input_df = input_df[
        features
    ]

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    for column in features:

        input_df[column] = pd.to_numeric(
            input_df[column],
            errors="raise"
        )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    probability = float(
        model.predict_proba(
            input_df
        )[0][1]
    )

    # --------------------------------------------------------
    # Risk classification
    # --------------------------------------------------------

    risk = classify_risk(
        probability,
        threshold
    )

    return {
        "probability": probability,
        "risk": risk,
        "threshold": threshold
    }


print(
    "Complete deployment prediction function created."
)

Complete deployment prediction function created.


In [18]:
# ============================================================
# CELL 17 - REAL DATASET PREDICTION TEST
# ============================================================

sample_row = df_features.iloc[0]

sample_input = {
    feature: float(
        sample_row[feature]
    )
    for feature in feature_columns
}

prediction_result = predict_landslide_risk(
    sample_input
)

print("=" * 70)
print("LANKALANDSLIDE AI - PREDICTION TEST")
print("=" * 70)

print()

print("Probability:")
print(
    f"{prediction_result['probability']:.6f}"
)

print()

print("Risk:")
print(
    prediction_result["risk"]
)

print()

print("Threshold:")
print(
    prediction_result["threshold"]
)

LANKALANDSLIDE AI - PREDICTION TEST

Probability:
0.005526

Risk:
LOW

Threshold:
0.49


In [19]:
# ============================================================
# CELL 18 - SECOND REAL DATASET TEST
# ============================================================

sample_row = df_features.iloc[1000]

sample_input = {
    feature: float(
        sample_row[feature]
    )
    for feature in feature_columns
}

prediction_result = predict_landslide_risk(
    sample_input
)

print("=" * 70)

print(
    "Probability:",
    f"{prediction_result['probability']:.6f}"
)

print(
    "Risk:",
    prediction_result["risk"]
)

print(
    "Threshold:",
    prediction_result["threshold"]
)

Probability: 0.000104
Risk: LOW
Threshold: 0.49


In [20]:
# ============================================================
# CELL 19 - BATCH PREDICTION TEST
# ============================================================

sample_data = X_all.head(
    100
)

sample_probabilities = final_model.predict_proba(
    sample_data
)[:, 1]

sample_risks = [
    classify_risk(
        probability,
        threshold
    )
    for probability in sample_probabilities
]

batch_results = pd.DataFrame({
    "probability": sample_probabilities,
    "risk": sample_risks
})

print(
    batch_results.head(
        20
    )
)

print()

print("Risk counts:")

print(
    batch_results["risk"].value_counts()
)

    probability risk
0      0.005526  LOW
1      0.006074  LOW
2      0.002184  LOW
3      0.000951  LOW
4      0.000813  LOW
5      0.000563  LOW
6      0.001191  LOW
7      0.001098  LOW
8      0.002152  LOW
9      0.001478  LOW
10     0.000544  LOW
11     0.002801  LOW
12     0.003247  LOW
13     0.001824  LOW
14     0.001075  LOW
15     0.000615  LOW
16     0.002153  LOW
17     0.000012  LOW
18     0.000026  LOW
19     0.000178  LOW

Risk counts:
risk
LOW    100
Name: count, dtype: int64


In [21]:
# ============================================================
# CELL 20 - SAVE DEPLOYMENT TEST RESULTS
# ============================================================

DEPLOYMENT_TEST_PATH = os.path.join(
    PROCESSED_DIR,
    "deployment_prediction_test.csv"
)

batch_results.to_csv(
    DEPLOYMENT_TEST_PATH,
    index=False
)

print(
    "Deployment test results saved:"
)

print(
    DEPLOYMENT_TEST_PATH
)

Deployment test results saved:
..\data\processed\deployment_prediction_test.csv


In [22]:
# ============================================================
# CELL 21 - MODEL INPUT RANGES
# ============================================================

input_statistics = []

for feature in feature_columns:

    series = pd.to_numeric(
        X_all[feature],
        errors="coerce"
    )

    input_statistics.append({
        "feature": feature,
        "minimum": float(series.min()),
        "maximum": float(series.max()),
        "mean": float(series.mean()),
        "median": float(series.median()),
        "missing": int(series.isna().sum())
    })

input_statistics_df = pd.DataFrame(
    input_statistics
)

print(
    input_statistics_df.to_string(
        index=False
    )
)

           feature    minimum     maximum       mean        median  missing
     rainfall_1day   0.000000  259.500000   5.107062  1.600000e+00        0
     rainfall_3day   0.000000  544.100000  15.321272  6.900000e+00        0
     rainfall_7day   0.000000  625.400000  35.750449  2.050000e+01        0
    rainfall_14day   0.000000  886.000000  71.454334  4.790000e+01        0
    rainfall_30day   0.000000 1498.900000 152.927163  1.182000e+02        0
 rainfall_3day_avg   0.000000  181.366667   5.107091  2.300000e+00        0
 rainfall_7day_avg   0.000000   89.342857   5.107207  2.928571e+00        0
rainfall_14day_avg   0.000000   63.285714   5.103881  3.421429e+00        0
rainfall_30day_avg   0.000000   49.963333   5.097572  3.940000e+00        0
        rain_hours   0.000000   24.000000   6.860931  5.000000e+00        0
          temp_max  12.700000   40.900000  30.062829  3.010000e+01        0
          temp_min   6.200000   30.200000  23.808471  2.430000e+01        0
         tem

In [23]:
# ============================================================
# CELL 22 - SAVE INPUT VALIDATION INFORMATION
# ============================================================

INPUT_STATS_PATH = os.path.join(
    MODEL_DIR,
    "input_statistics.json"
)

input_statistics_json = {}

for _, row in input_statistics_df.iterrows():

    input_statistics_json[
        row["feature"]
    ] = {
        "minimum": row["minimum"],
        "maximum": row["maximum"],
        "mean": row["mean"],
        "median": row["median"]
    }

with open(
    INPUT_STATS_PATH,
    "w"
) as f:

    json.dump(
        input_statistics_json,
        f,
        indent=4
    )

print(
    "Input validation statistics saved:"
)

print(
    INPUT_STATS_PATH
)

Input validation statistics saved:
..\models\input_statistics.json


In [24]:
# ============================================================
# CELL 23 - FINAL DEPLOYMENT PACKAGE VERIFICATION
# ============================================================

deployment_files = [
    "lankalandslide_final_model.pkl",
    "deployment_feature_columns.json",
    "deployment_config.json",
    "input_statistics.json"
]

print("=" * 70)
print("FINAL DEPLOYMENT PACKAGE")
print("=" * 70)

all_available = True

for filename in deployment_files:

    path = os.path.join(
        MODEL_DIR,
        filename
    )

    if os.path.exists(path):

        print(
            "✓",
            filename
        )

    else:

        print(
            "✗",
            filename
        )

        all_available = False

print()

if all_available:

    print(
        "✓ DEPLOYMENT PACKAGE READY"
    )

else:

    print(
        "✗ DEPLOYMENT PACKAGE INCOMPLETE"
    )

FINAL DEPLOYMENT PACKAGE
✓ lankalandslide_final_model.pkl
✓ deployment_feature_columns.json
✓ deployment_config.json
✓ input_statistics.json

✓ DEPLOYMENT PACKAGE READY


In [25]:
# ============================================================
# CELL 24 - DEPLOYMENT SUMMARY
# ============================================================

print("=" * 75)
print("LANKALANDSLIDE AI - DEPLOYMENT SUMMARY")
print("=" * 75)

print()

print("Final Model:")
print("XGBoost")

print()

print("Number of input features:")
print(len(feature_columns))

print()

print("Classification threshold:")
print(threshold)

print()

print("Training dataset:")
print(
    os.path.basename(
        FEATURE_DATA_PATH
    )
)

print()

print("Model file:")
print(
    "models/lankalandslide_final_model.pkl"
)

print()

print("Feature configuration:")
print(
    "models/deployment_feature_columns.json"
)

print()

print("Configuration:")
print(
    "models/deployment_config.json"
)

print()

print("Input statistics:")
print(
    "models/input_statistics.json"
)

print()

print("=" * 75)
print("MODEL IS READY FOR BACKEND INTEGRATION")
print("=" * 75)

LANKALANDSLIDE AI - DEPLOYMENT SUMMARY

Final Model:
XGBoost

Number of input features:
27

Classification threshold:
0.49

Training dataset:
landslide_weather_terrain_features.csv

Model file:
models/lankalandslide_final_model.pkl

Feature configuration:
models/deployment_feature_columns.json

Configuration:
models/deployment_config.json

Input statistics:
models/input_statistics.json

MODEL IS READY FOR BACKEND INTEGRATION
